# Two-Way ANOVA + Post-Hoc / Simple-Effects Analysis

This notebook demonstrates a **two-way ANOVA** for portfolio returns using:

- Factor 1: Trading Strategy
- Factor 2: Market Regime
- Dependent variable: Daily return

The analysis tests:

1. Strategy main effect
2. Market-regime main effect
3. Strategy × Market-regime interaction

If a main effect involving more than two levels is significant, Tukey HSD can be used for pairwise comparisons. If the interaction is significant, the notebook performs **simple-effects one-way ANOVA within each market regime** followed by Tukey HSD.

Decision rule:

- `p < 0.05` → Reject H₀
- `p >= 0.05` → Fail to reject H₀

In [ ]:
# Install if necessary
# %pip install pandas numpy statsmodels scipy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

alpha = 0.05

## 1. Create finance example data

We use three strategies observed under three market regimes.

The data below are illustrative daily returns and are intended for learning the procedure.

In [ ]:
rng = np.random.default_rng(42)

strategies = ["Strategy A", "Strategy B", "Strategy C"]
regimes = ["Bull", "Sideways", "Bear"]

strategy_effect = {
    "Strategy A": 0.12,
    "Strategy B": 0.10,
    "Strategy C": 0.07
}

regime_effect = {
    "Bull": 0.06,
    "Sideways": 0.00,
    "Bear": -0.05
}

rows = []

for strategy in strategies:
    for regime in regimes:
        mean = strategy_effect[strategy] + regime_effect[regime]
        returns = rng.normal(loc=mean, scale=0.04, size=30)

        for r in returns:
            rows.append({
                "Strategy": strategy,
                "Market_Regime": regime,
                "Return": r
            })

df = pd.DataFrame(rows)

df.head()

In [ ]:
# Check the number of observations in each cell
print(pd.crosstab(df["Strategy"], df["Market_Regime"]))

# Group means
group_means = (
    df.groupby(["Strategy", "Market_Regime"])["Return"]
      .mean()
      .unstack()
)

print("\nMean returns:")
display(group_means)

## 2. Fit the two-way ANOVA model

The formula is:

`Return ~ C(Strategy) + C(Market_Regime) + C(Strategy):C(Market_Regime)`

The interaction term is essential because it tests whether the effect of strategy changes across market regimes.

In [ ]:
model = ols(
    "Return ~ C(Strategy) + C(Market_Regime) + C(Strategy):C(Market_Regime)",
    data=df
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

display(anova_table)

## 3. Make decisions for the three effects

In [ ]:
effects = {
    "Strategy": "C(Strategy)",
    "Market Regime": "C(Market_Regime)",
    "Strategy × Market Regime": "C(Strategy):C(Market_Regime)"
}

decision_rows = []

for effect_name, effect_term in effects.items():
    p = anova_table.loc[effect_term, "PR(>F)"]

    if p < alpha:
        decision = "Reject H0"
        conclusion = "Statistically significant effect"
    else:
        decision = "Fail to reject H0"
        conclusion = "Insufficient evidence of an effect"

    decision_rows.append({
        "Effect": effect_name,
        "F-statistic": anova_table.loc[effect_term, "F"],
        "p-value": p,
        "Decision": decision,
        "Conclusion": conclusion
    })

decision_df = pd.DataFrame(decision_rows)
display(decision_df)

## 4. Tukey HSD for the Strategy main effect

There are three strategy levels, so if the **Strategy main effect** is significant, Tukey HSD can identify which strategy pairs differ.

Important: when the Strategy × Market-Regime interaction is significant, the overall Strategy main effect should be interpreted cautiously. The more useful follow-up is usually to examine strategy differences within each regime.

In [ ]:
strategy_p = anova_table.loc["C(Strategy)", "PR(>F)"]
interaction_p = anova_table.loc[
    "C(Strategy):C(Market_Regime)", "PR(>F)"
]

if strategy_p < alpha and interaction_p >= alpha:
    tukey_strategy = pairwise_tukeyhsd(
        endog=df["Return"],
        groups=df["Strategy"],
        alpha=alpha
    )

    print(tukey_strategy)
else:
    print(
        "Overall Strategy Tukey HSD is not automatically interpreted because "
        "either the Strategy main effect is not significant or the interaction is significant."
    )

## 5. Tukey HSD for Market Regime

There are three market-regime levels. If the Market-Regime main effect is significant and there is no significant interaction, Tukey HSD can identify which regimes differ.

In [ ]:
regime_p = anova_table.loc["C(Market_Regime)", "PR(>F)"]

if regime_p < alpha and interaction_p >= alpha:
    tukey_regime = pairwise_tukeyhsd(
        endog=df["Return"],
        groups=df["Market_Regime"],
        alpha=alpha
    )

    print(tukey_regime)
else:
    print(
        "Overall Market-Regime Tukey HSD is not automatically interpreted because "
        "either the Market-Regime main effect is not significant or the interaction is significant."
    )

## 6. If interaction is significant: simple-effects analysis

When Strategy × Market Regime is significant, ask:

> Within each market regime, do the strategy means differ?

For each regime, perform a one-way ANOVA across strategies. If significant, use Tukey HSD to identify the differing strategy pairs.

In [ ]:
def simple_effects_by_regime(data, alpha=0.05):
    results = []

    for regime in data["Market_Regime"].unique():
        subset = data[data["Market_Regime"] == regime]

        model_regime = ols(
            "Return ~ C(Strategy)",
            data=subset
        ).fit()

        anova_regime = sm.stats.anova_lm(model_regime, typ=2)
        p = anova_regime.loc["C(Strategy)", "PR(>F)"]

        if p < alpha:
            decision = "Reject H0"
            conclusion = "Strategy means differ within this regime"

            tukey = pairwise_tukeyhsd(
                endog=subset["Return"],
                groups=subset["Strategy"],
                alpha=alpha
            )
        else:
            decision = "Fail to reject H0"
            conclusion = "Insufficient evidence that strategy means differ within this regime"
            tukey = None

        results.append({
            "Regime": regime,
            "F-statistic": anova_regime.loc["C(Strategy)", "F"],
            "p-value": p,
            "Decision": decision,
            "Conclusion": conclusion,
            "Tukey": tukey
        })

    return results

if interaction_p < alpha:
    simple_results = simple_effects_by_regime(df, alpha)

    for result in simple_results:
        print("=" * 70)
        print("Market Regime:", result["Regime"])
        print("F-statistic:", round(result["F-statistic"], 4))
        print("p-value:", round(result["p-value"], 6))
        print("Decision:", result["Decision"])
        print("Conclusion:", result["Conclusion"])

        if result["Tukey"] is not None:
            print("\nTukey HSD:")
            print(result["Tukey"])
else:
    print("Interaction is not significant; simple-effects analysis is not required.")

## 7. Final interpretation framework

Use the following workflow:

### Step 1 — Strategy main effect
- `p < 0.05` → strategy has a statistically significant effect on mean return.
- `p >= 0.05` → insufficient evidence of a strategy effect.

### Step 2 — Market-regime main effect
- `p < 0.05` → market regime has a statistically significant effect.
- `p >= 0.05` → insufficient evidence of a regime effect.

### Step 3 — Interaction
- `p < 0.05` → strategy performance depends on market regime.
- `p >= 0.05` → insufficient evidence of an interaction.

### Step 4 — Post-hoc
If a significant effect has more than two levels, use Tukey HSD to determine which pairs differ.

If the interaction is significant, examine **simple effects within each regime** rather than relying only on the overall main effects.

## 8. Important assumptions

Standard two-way ANOVA assumes:

1. Independent observations.
2. A continuous dependent variable.
3. Approximately normally distributed residuals.
4. Reasonably homogeneous residual variance across cells.
5. Correct specification of the factors and interaction.

For financial returns, independence and constant variance can be problematic because returns may show autocorrelation or volatility clustering. Therefore, the standard ANOVA result should be validated with appropriate time-series diagnostics and robust/resampling methods when necessary.

Also, statistical significance does not automatically mean an investment strategy is economically superior. Examine effect size, volatility, Sharpe ratio, drawdown, transaction costs, turnover, and out-of-sample performance.